[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/skarma91/logicmojo-ai-july-2026/blob/main/modules/module-1-python-for-ai/03-python-for-real-work/code/classes_error_handling_files.ipynb)

# Class 1.3: Python for real work

The slides carry the ideas. Here you run and tweak the code.

**What we will cover**

- Classes: `__init__`, attributes, instance methods, and `self`
- Objects with independent state, and `type()`
- Instance vs class vs static methods
- Inheritance, `super()`, and overriding
- Polymorphism (duck typing)
- Multiple inheritance and the MRO (awareness)
- Errors: multiple `except` blocks, and `try` / `except` / `else` / `finally`
- Files: plain text, then JSON
- Build: a small class-based JSON tool

Run each cell, change a value, and run it again.

## Classes

A class bundles data (attributes) with the actions on it (methods). `self` is the object the method is working on.

In [1]:
class Account:
    def __init__(self, balance):
        self.balance = balance       # attribute

    def deposit(self, amount):       # instance method
        self.balance += amount

a = Account(100)
b = Account(0)
a.deposit(50)

print("a:", a.balance, "| b:", b.balance)   # independent state
print("type:", type(a))                     # <class '__main__.Account'>

a: 150 | b: 0
type: <class '__main__.Account'>


## Instance, class, and static methods

Three kinds of method: one bound to the object (`self`), one bound to the class (`cls`), and a plain helper grouped with the class.

In [ ]:
class User:
    def __init__(self, name, age):
        self.name = name
        self.age = age

    def greeting(self):                  # instance method
        return f"Hi, I am {self.name}"

    @classmethod
    def from_row(cls, row):              # class method: build from a "name,age" string
        name, age = row.split(",")
        return cls(name, int(age))

    @staticmethod
    def is_adult(age):                   # static method: a plain helper
        return age >= 18

u = User.from_row("Ada,36")
print(u.greeting())
print("adult?", User.is_adult(u.age))

Hi, I am Ada
adult? True


## Inheritance

A subclass reuses a parent and extends it. `super()` calls the parent's version; overriding a method changes the behavior.

In [4]:
class Account:
    def __init__(self, balance):
        self.balance = balance
    def summary(self):
        return f"balance {self.balance}"

class SavingsAccount(Account):
    def __init__(self, balance, rate):
        super().__init__(balance)    # reuse the parent __init__
        self.rate = rate
    def summary(self):               # override
        return f"balance {self.balance}, rate {self.rate}"

s = SavingsAccount(100, 0.05)
print(s.summary())
print(s.balance)   # 100, set by the parent via super()

balance 100, rate 0.05
100


## Polymorphism

The same method name runs different code depending on the object. The caller does not need to know the exact type (duck typing).

In [5]:
for acc in [Account(100), SavingsAccount(100, 0.05)]:
    print(f"{type(acc).__name__}: {acc.summary()}")

Account: balance 100
SavingsAccount: balance 100, rate 0.05


## Multiple inheritance and the MRO (use sparingly)

A class can have more than one parent. When both define the same method, Python follows the **method resolution order** (MRO), left to right. Prefer single inheritance or small mixins; this gets subtle fast.

In [6]:
class Walk:
    def move(self): return "walking"
class Swim:
    def move(self): return "swimming"

class Amphibian(Walk, Swim):
    pass

a = Amphibian()
print(a.move())   # walking: Walk comes first
print([c.__name__ for c in Amphibian.__mro__])

walking
['Amphibian', 'Walk', 'Swim', 'object']


## Handling errors: catch each type

Some failures are expected. `try`/`except` lets you respond instead of crashing. Give each failure its own `except`, listing specific errors before general ones, and group errors that share a response with a tuple: `except (A, B) as e`.

In [8]:
# separate handlers: each failure gets its own response
def lookup(prices, item):
    try:
        return f"{item} costs {prices[item]}"
    except KeyError:
        return f"no price for {item}"
    # except TypeError:
    #     return "prices must be a dict"
    except Exception as e:
        return f"unexpected error: {e}"

menu = {"tea": 3, "coffee": 4}
print(lookup(menu, "tea"))       # found
print(lookup(menu, "cocoa"))     # KeyError
print(lookup((("Tea", 3), ("Coffee", 4)), "tea"))       # TypeError

tea costs 3
no price for cocoa
unexpected error: tuple indices must be integers or slices, not str


In [9]:
# one handler for several related errors, and 'as e' to inspect it
for value in ["42", None, "x"]:
    try:
        print("parsed:", int(value))
    except (ValueError, TypeError) as e:
        print(f"skip {value!r}: {type(e).__name__}")

parsed: 42
skip None: TypeError
skip 'x': ValueError


## try / except / else / finally

The full shape. `else` runs only when the `try` block raised nothing, so keep just the risky line in `try` and the follow-up in `else`. `finally` runs no matter what (success, handled error, or an error on the way out), which makes it the place for cleanup. Notice `finally` even runs when the branches `return`.

In [10]:
def withdraw(balance, amount):
    try:
        amount = int(amount)              # the only risky step
    except ValueError:
        print("  amount must be a number")
        return balance
    else:
        # runs only when the try above raised nothing
        if amount > balance:
            print("  insufficient funds")
        else:
            balance -= amount
            print(f"  withdrew {amount}")
        return balance
    finally:
        print("  -- transaction logged --")   # always runs

bal = 100
print("try '30':")
bal = withdraw(bal, "30")     # else path
print("try 'oops':")
bal = withdraw(bal, "oops")   # except path
print("final balance:", bal)

try '30':
  withdrew 30
  -- transaction logged --
try 'oops':
  amount must be a number
  -- transaction logged --
final balance: 70


## Text files

`with open(...)` opens a file and closes it automatically. Mode sets the intent: `"r"` read (default), `"w"` overwrite, `"a"` append. `.read()` returns the whole file as one string; looping over the file hands you one line at a time, which scales to files too big to hold in memory.

In [ ]:
import os

# Write a small log, then read it back. A text file is just characters;
# any structure (here, a level and a message per line) is ours to impose.
with open("run.log", "w") as f:
    f.write("INFO service started\n")
    f.write("WARN disk almost full\n")
    f.write("INFO request handled\n")
    f.write("WARN retry limit reached\n")

# 1) read the whole file as one string
with open("run.log") as f:
    print(f.read())

# 2) process it line by line and keep only the warnings
with open("run.log") as f:
    warnings = [line.strip() for line in f if line.startswith("WARN")]

print("warnings:", warnings)
print("count:", len(warnings))

os.remove("run.log")   # clean up the demo file

warnings: ['WARN disk almost full', 'WARN retry limit reached']
count: 2


## JSON

When the data has structure, save it as JSON rather than hand-formatted text. A JSON object maps straight onto a Python dict, and the types survive the round trip (a number comes back a number, not a string).

In [16]:
import json

data = {"course": "GenAI", "classes": 45}
text = json.dumps(data)              # dict -> JSON text
print("as text:", text)
print("back to dict:", json.loads(text)["classes"])

as text: {"course": "GenAI", "classes": 45}
back to dict: 45


In [19]:
course_data = {"Instructor": "Sourav", "Course": "GenAI", "Classes": 45, "Topics": ["Python", "AI", "GenAI"], 
               "Module-1": {"Name": "Python for AI", "Duration": "2 weeks"},
               "Module-2": {"Name": "GenAI", "Duration": "3 weeks"},
               "Module-3": {"Name": "AI Ethics", "Duration": "1 week"},
               "is_live": True}

with open("course_data.json", "w") as f:
    json.dump(course_data, f, indent=4)                # dict -> JSON file

In [21]:
with open("course_data.json") as f:
    loaded_data = json.load(f)                         # JSON file -> dict

print("loaded data:", loaded_data)
print(type(loaded_data))

loaded data: {'Instructor': 'Sourav', 'Course': 'GenAI', 'Classes': 45, 'Topics': ['Python', 'AI', 'GenAI'], 'Module-1': {'Name': 'Python for AI', 'Duration': '2 weeks'}, 'Module-2': {'Name': 'GenAI', 'Duration': '3 weeks'}, 'Module-3': {'Name': 'AI Ethics', 'Duration': '1 week'}, 'is_live': True}
<class 'dict'>


## Build: a small class-based JSON tool

Reads a JSON file, transforms the records, and writes JSON out, with `try`/`except` so a missing file is handled cleanly. In a real project this class would live in its own file and be imported (`from tool import RecordTool`). The demo files are removed at the end.

In [ ]:
import json
import os

class RecordTool:
    def __init__(self, path):
        self.path = path

    def load(self):
        try:
            with open(self.path) as f:
                return json.load(f)
        except FileNotFoundError:
            print(f"no file at {self.path}; returning an empty list")
            return []

    def save(self, records):
        with open(self.path, "w") as f:
            json.dump(records, f, indent=2)
        f.close()

people = RecordTool("people.json")
people.save([
    {"name": "Ada", "age": 36},
    {"name": "Alan", "age": 41},
    {"name": "Grace", "age": 28},
])

rows = people.load()
enriched = []
for p in rows:
    enriched.append({"name": p["name"], "age": p["age"], "senior": p["age"] >= 40})

out = RecordTool("adults.json")
out.save(enriched)
print(out.load())

print(RecordTool("nope.json").load())   # try/except path: missing file

os.remove("people.json")
os.remove("adults.json")

[{'name': 'Ada', 'age': 36, 'senior': False}, {'name': 'Alan', 'age': 41, 'senior': True}, {'name': 'Grace', 'age': 28, 'senior': False}]
no file at nope.json; returning an empty list
[]


## Your turn

**Micro-assignment.** Five problems on classes (including inheritance and polymorphism), errors, and JSON; see `../micro-assignment/README.md`.

**Next, class 1.4 (The data stack):** NumPy arrays and Pandas tables.